# Mr Predictor: resilient modulus from any available properties

Predicts **RESILIENT_MODULUS (Mr)** from any combination of these inputs:

`MATL_CODE_EXP, LIQUID_LIMIT, PLASTIC_LIMIT, PLASTICITY_INDEX, AASHTO_SOIL_CLASS_EXP, SPEC_GRAVITY, MAX_LAB_DRY_DENSITY, OPTIMUM_LAB_MOISTURE, RESILIENT_MODULUS_K1, RESILIENT_MODULUS_K2, RESILIENT_MODULUS_K3, POISSONS_RATIO, HYDRAULIC_CONDUCTIVITY`, plus the optional extras `REPR_THICKNESS` and `LAYER_TYPE_EXP`.

The notebook trains a gradient-boosting model on **exactly the inputs you give**. With all values, every property is used; with only some values, only those are used.

**How to run:** Runtime → Run all. Paste your Drive link in step 1 (or upload the file when asked), then edit the values in step 4, or upload your own file of values in step 6.

## 1. Load the data from Google Drive

1. Upload **`Mr Data.xlsx`** (the LTPP sheet with `RESILIENT_MODULUS`) to your Google Drive.
2. Right-click → **Share** → **Copy link**, and paste it into `DATA_LINK` below (already set to your file).
3. If the file is shared as *Restricted*, Colab will ask you to sign in to Google and allow access. That's normal. Choose the account that owns the file. To skip this, set General access to **Anyone with the link**.

Set `DATA_LINK = ""` to choose the file from your computer instead.

In [ ]:
import os, re, urllib.request
import pandas as pd

# Google Drive share link of Mr Data.xlsx (private files ask you to sign in to Google)
DATA_LINK = "https://docs.google.com/spreadsheets/d/1aid0EDrUKtc_8ceqRdklLo-3D0AFRgeP/edit?usp=drive_link&ouid=115497790907205429656&rtpof=true&sd=true"
DATA_FILE = "Mr Data.xlsx"

def download_from_drive(link, dest):
    m = re.search(r"/d/([\w-]+)", link) or re.search(r"[?&]id=([\w-]+)", link)
    if not m:
        raise ValueError("Could not find a file id in DATA_LINK.")
    fid = m.group(1)
    urls = [f"https://drive.google.com/uc?export=download&id={fid}",                   # uploaded .xlsx
            f"https://docs.google.com/spreadsheets/d/{fid}/export?format=xlsx"]        # native Google Sheet
    for url in urls:
        try:
            data = urllib.request.urlopen(url).read()
        except Exception:
            continue
        if data[:2] == b"PK":   # xlsx files are zip archives
            open(dest, "wb").write(data)
            return
    # Private file: sign in with the Google account that can open it
    try:
        from google.colab import auth
    except ImportError:
        raise RuntimeError("Download failed. Share the file as 'Anyone with the link' or run in Colab.")
    import io
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    auth.authenticate_user()
    svc = build("drive", "v3")
    meta = svc.files().get(fileId=fid, fields="mimeType", supportsAllDrives=True).execute()
    if meta["mimeType"] == "application/vnd.google-apps.spreadsheet":
        req = svc.files().export_media(fileId=fid, mimeType="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet")
    else:
        req = svc.files().get_media(fileId=fid, supportsAllDrives=True)
    with io.FileIO(dest, "wb") as fh:
        dl, done = MediaIoBaseDownload(fh, req), False
        while not done:
            _, done = dl.next_chunk()

if DATA_LINK:
    download_from_drive(DATA_LINK, DATA_FILE)
elif not os.path.exists(DATA_FILE):
    from google.colab import files
    print("Choose 'Mr Data.xlsx' from your computer...")
    name = next(iter(files.upload()))
    if name != DATA_FILE:
        os.rename(name, DATA_FILE)

header = pd.read_excel(DATA_FILE, nrows=0).columns
if "RESILIENT_MODULUS" not in header:
    raise ValueError("This file is not the Mr data sheet (no RESILIENT_MODULUS column). "
                     "Check DATA_LINK points to Mr Data.xlsx.")
print("Loaded", DATA_FILE)

## 2. Model code (no need to edit)

In [ ]:
import hashlib
import json
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold

HERE = os.getcwd()  # /content in Colab
DATA_XLSX = DATA_FILE
TRAIN_DATA = os.path.join(HERE, "mr_training_data.csv")
CACHE_DIR = os.path.join(HERE, "model_cache")

TARGET = "RESILIENT_MODULUS"
CATEGORICAL = ["MATL_CODE_EXP", "AASHTO_SOIL_CLASS_EXP", "LAYER_TYPE_EXP"]
# Columns requested for the model, in the order of the data sheet.
FEATURES = [
    "MATL_CODE_EXP", "LIQUID_LIMIT", "PLASTIC_LIMIT", "PLASTICITY_INDEX",
    "AASHTO_SOIL_CLASS_EXP", "SPEC_GRAVITY", "MAX_LAB_DRY_DENSITY",
    "OPTIMUM_LAB_MOISTURE", "RESILIENT_MODULUS_K1", "RESILIENT_MODULUS_K2",
    "RESILIENT_MODULUS_K3", "POISSONS_RATIO", "HYDRAULIC_CONDUCTIVITY",
]
# Optional extras: Mr in this data also depends on the layer (thickness/position).
EXTRA_FEATURES = ["REPR_THICKNESS", "LAYER_TYPE_EXP"]
ALL_FEATURES = FEATURES + EXTRA_FEATURES

GROUP_COLS = ["STATE_CODE", "SHRP_ID"]  # keep a test section's rows in one CV fold


def build_training_data(xlsx_path=DATA_XLSX, out_path=TRAIN_DATA):
    """Extract the rows with a measured Mr from the Excel sheet and save them."""
    df = pd.read_excel(xlsx_path)
    df = df[df[TARGET].notna()]
    df = df[GROUP_COLS + ALL_FEATURES + [TARGET]].reset_index(drop=True)
    for c in CATEGORICAL:
        df[c] = df[c].astype("object").where(df[c].notna(), None)
        df[c] = df[c].map(lambda v: None if v is None else str(v).strip())
    df.to_csv(out_path, index=False)
    return df


def _norm(text):
    return " ".join(str(text).lower().split())


class MrPredictor:
    def __init__(self, train_path=TRAIN_DATA, cache_dir=CACHE_DIR):
        if not os.path.exists(train_path):
            build_training_data(out_path=train_path)
        self.data = pd.read_csv(train_path, dtype={c: str for c in GROUP_COLS + CATEGORICAL})
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)
        self.categories = {
            c: sorted(self.data[c].dropna().unique()) for c in CATEGORICAL
        }
        self._lookup = {c: {_norm(v): v for v in cats} for c, cats in self.categories.items()}
        self._models = {}

    # ---------- input handling ----------
    def clean_inputs(self, inputs):
        """Keep the known, non-empty inputs; returns (values, warnings)."""
        aliases = {_norm(f): f for f in ALL_FEATURES}
        values, warnings = {}, []
        for key, val in inputs.items():
            col = aliases.get(_norm(key))
            if col is None:
                warnings.append(f"Ignored unknown input '{key}'.")
                continue
            if val is None or (isinstance(val, float) and np.isnan(val)) or str(val).strip() == "":
                continue
            if col in CATEGORICAL:
                match = self._lookup[col].get(_norm(val))
                if match is None:
                    warnings.append(f"'{val}' is not a known {col} value; it was ignored.")
                    continue
                values[col] = match
            else:
                try:
                    values[col] = float(val)
                except ValueError:
                    warnings.append(f"{col} must be a number (got '{val}'); it was ignored.")
        if "POISSONS_RATIO" in values:
            warnings.append("POISSONS_RATIO is 0.35 for every training row, so it cannot affect Mr.")
        return values, warnings

    def _encode(self, frame, cols):
        X = pd.DataFrame(index=frame.index)
        for c in cols:
            if c in CATEGORICAL:
                codes = {v: i for i, v in enumerate(self.categories[c])}
                X[c] = frame[c].map(codes)
            else:
                X[c] = frame[c]
        return X.astype(float)

    # ---------- model per input subset ----------
    def _new_model(self, cols):
        return HistGradientBoostingRegressor(
            max_iter=600, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=10,
            l2_regularization=1.0, categorical_features=[c in CATEGORICAL for c in cols],
            random_state=0,
        )

    def get_model(self, cols):
        """Model trained on exactly ``cols`` (cached in memory and on disk)."""
        cols = [c for c in ALL_FEATURES if c in cols]
        key = hashlib.md5(json.dumps(cols).encode()).hexdigest()[:12]
        if key in self._models:
            return self._models[key]
        path = os.path.join(self.cache_dir, f"mr_{key}.joblib")
        if os.path.exists(path):
            entry = joblib.load(path)
        else:
            entry = self._train(cols)
            joblib.dump(entry, path)
        self._models[key] = entry
        return entry

    def _train(self, cols):
        d = self.data
        X, y = self._encode(d, cols), np.log(d[TARGET].to_numpy())
        groups = d[GROUP_COLS].astype(str).agg("_".join, axis=1)
        # 5-fold cross-validation grouped by test section -> expected error
        pred = np.empty_like(y)
        for tr, te in GroupKFold(n_splits=5).split(X, y, groups):
            pred[te] = self._new_model(cols).fit(X.iloc[tr], y[tr]).predict(X.iloc[te])
        true, est = np.exp(y), np.exp(pred)
        log_resid = y - pred
        cv = {
            "R2": float(1 - ((true - est) ** 2).sum() / ((true - true.mean()) ** 2).sum()),
            "MAE": float(np.abs(true - est).mean()),
            "MAPE_%": float(100 * np.mean(np.abs(true - est) / true)),
            # 10th/90th percentile of the CV error ratio -> ~80% prediction band
            "band_lo": float(np.exp(np.quantile(log_resid, 0.10))),
            "band_hi": float(np.exp(np.quantile(log_resid, 0.90))),
        }
        model = self._new_model(cols).fit(X, y)
        return {"cols": cols, "model": model, "cv": cv}

    # ---------- prediction ----------
    def predict(self, **inputs):
        values, warnings = self.clean_inputs(inputs)
        if not values:
            raise ValueError("Give at least one input value.")
        entry = self.get_model(list(values))
        X = self._encode(pd.DataFrame([values]), entry["cols"])
        mr = float(np.exp(entry["model"].predict(X)[0]))
        cv = entry["cv"]
        return {
            "RESILIENT_MODULUS": mr,
            "range_80pct": (mr * cv["band_lo"], mr * cv["band_hi"]),
            "inputs_used": values,
            "cv_metrics": {k: cv[k] for k in ("R2", "MAE", "MAPE_%")},
            "warnings": warnings,
        }

    def predict_table(self, frame):
        """Predict Mr for every row of a DataFrame; blank cells count as not given.

        Columns that are not model inputs (IDs, measured Mr, ...) are kept as they are.
        """
        inputs = {_norm(f) for f in ALL_FEATURES}
        rows = []
        for _, row in frame.iterrows():
            given = {k: v for k, v in row.items() if _norm(k) in inputs}
            try:
                r = self.predict(**given)
                rows.append({
                    "PREDICTED_MR": r["RESILIENT_MODULUS"],
                    "MR_LOW_80": r["range_80pct"][0], "MR_HIGH_80": r["range_80pct"][1],
                    "INPUTS_USED": ", ".join(r["inputs_used"]),
                    "MODEL_CV_R2": r["cv_metrics"]["R2"],
                    "MODEL_CV_MAE": r["cv_metrics"]["MAE"],
                    "NOTES": " ".join(r["warnings"]),
                })
            except ValueError as e:
                rows.append({"PREDICTED_MR": np.nan, "INPUTS_USED": str(e)})
        return pd.concat([frame.reset_index(drop=True), pd.DataFrame(rows)], axis=1)

## 3. Load data and check accuracy
The first run takes about 20 seconds. The table shows the cross-validated accuracy for common input combinations.

In [ ]:
build_training_data()          # reads DATA_FILE, keeps the rows that have a measured Mr
p = MrPredictor()
print("Training rows with measured Mr:", len(p.data))

checks = {
    "All requested properties": FEATURES,
    "All + thickness/layer type": FEATURES + EXTRA_FEATURES,
    "K1, K2, K3 only": ["RESILIENT_MODULUS_K1", "RESILIENT_MODULUS_K2", "RESILIENT_MODULUS_K3"],
    "Soil properties only (no K)": ["MATL_CODE_EXP", "LIQUID_LIMIT", "PLASTIC_LIMIT", "PLASTICITY_INDEX",
                                    "AASHTO_SOIL_CLASS_EXP", "SPEC_GRAVITY", "MAX_LAB_DRY_DENSITY",
                                    "OPTIMUM_LAB_MOISTURE", "HYDRAULIC_CONDUCTIVITY"],
}
pd.DataFrame({name: p.get_model(cols)["cv"] for name, cols in checks.items()}).T[["R2", "MAE", "MAPE_%"]].round(3)

## 4. Predict Mr: edit the values and run

Put `None` for any value you don't have. Text values must match the names in the sheet; run step 5 to list them. To predict many samples from your own Excel/CSV file, use step 6. A new combination of inputs takes about 5 seconds the first time.

In [ ]:
inputs = {
    "MATL_CODE_EXP":          "Crushed Stone",
    "LIQUID_LIMIT":           0,
    "PLASTIC_LIMIT":          0,
    "PLASTICITY_INDEX":       None,
    "AASHTO_SOIL_CLASS_EXP":  "A-1-a",
    "SPEC_GRAVITY":           2.656,
    "MAX_LAB_DRY_DENSITY":    135,
    "OPTIMUM_LAB_MOISTURE":   7,
    "RESILIENT_MODULUS_K1":   900.411,
    "RESILIENT_MODULUS_K2":   0.626,
    "RESILIENT_MODULUS_K3":   -0.123,
    "POISSONS_RATIO":         None,
    "HYDRAULIC_CONDUCTIVITY": 0.05054,
    # optional extras
    "REPR_THICKNESS":         None,
    "LAYER_TYPE_EXP":         None,
}

def show(result):
    lo, hi = result["range_80pct"]
    cv = result["cv_metrics"]
    for w in result["warnings"]:
        print("note:", w)
    print(f"Predicted RESILIENT_MODULUS = {result['RESILIENT_MODULUS']:.2f}")
    print(f"Likely range (80%)          = {lo:.2f} - {hi:.2f}")
    print(f"Inputs used ({len(result['inputs_used'])}): {', '.join(result['inputs_used'])}")
    print(f"Accuracy for this input combination: R2={cv['R2']:.2f}, MAE={cv['MAE']:.2f}, MAPE={cv['MAPE_%']:.1f}%")

show(p.predict(**inputs))

## 5. Allowed text values
Use these exact names for the text columns (capital letters don't matter).

In [ ]:
for col in CATEGORICAL:
    print()
    print(f"{col}:")
    for c in p.categories[col]:
        print("  ", c)

## 6. Predict your own values from a file

**6a. Get the template:** tick the box and run the cell. `Mr_input_template.xlsx` downloads with every input column, two example rows, and an *Allowed values* sheet.

**6b. Fill it in:** one sample per row. Leave a cell **blank** if you don't have that value. You can delete columns you never use, add your own columns (e.g. a sample ID), and if you include a `RESILIENT_MODULUS` column of measured values it is kept next to the prediction for comparison. A `.csv` file with the same column names also works.

**6c. Upload and predict:** run the last cell and choose your file. The results appear below it, and `<your file>_predicted.xlsx` downloads.

In [ ]:
#@title 6a. Download the input template
DOWNLOAD_TEMPLATE = False  #@param {type:"boolean"}

if DOWNLOAD_TEMPLATE:
    from google.colab import files
    example_full = {**{c: None for c in ALL_FEATURES},
                    "MATL_CODE_EXP": "Crushed Stone", "LIQUID_LIMIT": 0, "PLASTIC_LIMIT": 0,
                    "AASHTO_SOIL_CLASS_EXP": "A-1-a", "SPEC_GRAVITY": 2.656, "MAX_LAB_DRY_DENSITY": 135,
                    "OPTIMUM_LAB_MOISTURE": 7, "RESILIENT_MODULUS_K1": 900.411, "RESILIENT_MODULUS_K2": 0.626,
                    "RESILIENT_MODULUS_K3": -0.123, "HYDRAULIC_CONDUCTIVITY": 0.05054}
    example_partial = {**{c: None for c in ALL_FEATURES},
                       "MATL_CODE_EXP": "Fine-Grained Soils: Clay", "LIQUID_LIMIT": 45, "PLASTIC_LIMIT": 20,
                       "MAX_LAB_DRY_DENSITY": 105}
    template = pd.DataFrame([example_full, example_partial], columns=["SAMPLE_ID"] + ALL_FEATURES)
    template["SAMPLE_ID"] = ["example (all values)", "example (only some values)"]
    allowed = pd.concat({c: pd.Series(p.categories[c]) for c in CATEGORICAL}, axis=1)
    with pd.ExcelWriter("Mr_input_template.xlsx") as xw:
        template.to_excel(xw, sheet_name="Inputs", index=False)
        allowed.to_excel(xw, sheet_name="Allowed values", index=False)
    files.download("Mr_input_template.xlsx")
else:
    print("Tick DOWNLOAD_TEMPLATE and run this cell to get the template.")

In [ ]:
#@title 6c. Upload your values file and predict
from google.colab import files

uploaded = files.upload()   # choose your .xlsx or .csv file
if not uploaded:
    print("No file chosen. Run this cell again to upload.")
else:
    name = next(iter(uploaded))
    frame = pd.read_excel(name) if name.lower().endswith((".xlsx", ".xls")) else pd.read_csv(name)
    frame = frame.dropna(how="all")
    print(f"Predicting {len(frame)} rows from {name} ...")
    out = p.predict_table(frame)
    out_name = os.path.splitext(name)[0] + "_predicted.xlsx"
    out.to_excel(out_name, index=False)
    show_cols = [c for c in ["SAMPLE_ID", "RESILIENT_MODULUS"] if c in out] +                 ["PREDICTED_MR", "MR_LOW_80", "MR_HIGH_80", "MODEL_CV_R2", "INPUTS_USED", "NOTES"]
    display(out[[c for c in show_cols if c in out]].round(3))
    files.download(out_name)